# Exercise 01 — Workbook: your own protein

This notebook is **deliberately almost empty.** Its companion, **`ex01_guide.ipynb`**,
walks the entire analysis through on one protein (1FSZ, FtsZ) with every cell filled in.
Here you rebuild that analysis on a protein of **your own** choosing.

**Read `ex01_guide.ipynb` first.** Then work here, with it open beside you.

## How to work in this notebook

- **Copy code across from the guide and adapt it.** `renumber_to_uniprot`, `rcsb_chain`,
  `search_pdb`, `structure_table`, and the alignment helpers
  `hmmer_search`, `align`, `drop_redundant` and `conservation` are all written to be
  generic: they work on any protein, not just 1FSZ. Reusing them is the point.
- **Structure the notebook however you like.** The section headers below are a checklist
  of what has to be here, not a cell-by-cell template. Add cells, split them, reorder
  them, delete these prompts once you've answered them.
- **The reasoning is what's graded, not the code.** Every section below asks for a
  written conclusion as well as output. A notebook full of correct plots with no
  interpretation scores badly; one with an honest, well-argued interpretation of a
  messy result scores well.

## Using AI Tools

You may use AI assistants (ChatGPT, Claude, etc.) for syntax, debugging, and code
snippets. You must supply the biological reasoning, the justification for your choices,
and the critical evaluation of what came back. **"The AI said so" is not an answer.**

**Note:** two prompts below (B-factor flexibility, and conserved regions) are
predict-first: you write your prediction down *before* you run anything. That written
prediction is the graded artefact. Being wrong costs you nothing; not having made a
prediction costs you the marks.

## Setup

Same environment as the guide — run this first.

In [ ]:
# Check if running on Google Colab
try:
    from google.colab import drive  # noqa: F401  (import is the availability test)
    is_google_colab = True
except ImportError:
    is_google_colab = False

# If on Google Colab, install the package
if is_google_colab:
    %pip install numpy==2.1.3 scipy==1.16.3 pandas==2.2.3 biopandas==0.4.1 py3dmol==2.4.0 biopython==1.85 pyfamsa==0.7.0 pymsaviz==0.5.0

# NOTE: Ignore specific warning message from ipykernel=5.5.6
import warnings
import os

In [ ]:
# Import libraries
import gzip
import io
import time
import xml.etree.ElementTree as ET
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from biopandas.pdb import PandasPdb
from Bio import SeqIO
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils import seq1
from pyfamsa import Aligner, Sequence
from pymsaviz import MsaViz
import py3Dmol


# Suppress all warnings at the Python level
warnings.filterwarnings('ignore')

# Also set environment variable to suppress warnings
os.environ['PYTHONWARNINGS'] = 'ignore'

# pandas draws through matplotlib by default -- df.plot() gives us a real Axes back,
# which keeps most plots in this notebook to two or three lines.
print("All libraries loaded successfully")

---

## Character Sheet — your capstone protein

You **choose** your protein from the candidate list in `candidate_proteins.ipynb`, linked
from the README. **The ligand is not a choice**: each candidate is listed with the one
ligand that has already been run end to end with that protein, through molecular dynamics,
docking and a simulation of the two together. Pick a different het group out of the same
crystal and the later exercises will fail, or run and give you a meaningless number.

That pairing is the spine of the course: the same protein and ligand run through ex02
(AlphaFold), ex03 (molecular dynamics), ex04 (docking), and your final project. Choose one
you will be happy to live with for four days.

Record your choice:

- **PDB ID:**
- **Protein name:**
- **Organism:**
- **Method and resolution** (X-ray? NMR? cryo-EM? at what resolution?):
- **Ligand (CCD code and name):**


### Things to take care of

**Residue numbers.** A PDB file numbers its residues the way its authors chose, and often
not the way UniProt does: a signal peptide or the first methionine left uncounted, a tag
counted in, a residue deleted from the construct. UniProt's sites, AlphaFold's model and
AlphaMissense (ex02) all count in UniProt's numbers. Where your file disagrees, every
comparison by residue number pairs the wrong residues, and nothing warns you.

- **Before anything else**, run `renumber_to_uniprot` on your entry (the guide, end of
  section 1). It prints what it changed.
- **If it changed anything, work on its file from then on**: `load_structure(path)`,
  `show_structure(path, ...)` and `rcsb_chain(pdb_id, chain, numbering[chain])`. Read your
  contact residues off that file too.
- **Use the same file in ex02, ex03 and ex04.** Colab forgets files between notebooks:
  download it from the Files panel, or copy the helper along and run it again.
- **When numbers stop lining up, suspect this first**: an AlphaFold comparison far worse
  than about 1 Å, a site or a conserved residue on the wrong amino acid, contact lists that
  disagree between exercises, residue numbers above 10000 after ex03's preparation.

### Now catalogue *everything* in the structure

The candidate list names **one** ligand for your protein. Your structure almost certainly contains **more than
one** non-protein species, and telling them apart is the point of this section.

List **every** heteroatom species in your entry — the way the guide does for 1FSZ — with
how many atoms and how many copies of each. Then classify each one:

| Code | Name | Atoms | Copies | Biological, or experimental artifact? | Evidence |
|---|---|---|---|---|---|
| | | | | | |

**"Evidence" is the graded column.** For each species, say *how you decided*, using more
than one line of reasoning:

- What does the **RCSB entry page or the paper** say the structure was solved to study?
- **Where does it sit** — buried in a pocket making several contacts, or perched in a
  surface groove?
- Do you **recognise it as a lab reagent**? (glycerol, ethylene glycol, HEPES, MES, PEG,
  sulfate, acetate, DMSO…)
- Does it **recur in related structures** of the same protein, or appear only here?
- What do its **occupancy and B-factors** suggest about how well-ordered it is?

**Then find the binding site of your chosen ligand.**

**Look first.** `show_structure(your_pdb, ligand="XXX", pocket_of="XXX")` asks the viewer
itself for the residues within 4 Å of that species and labels them, using the same `within`
and `byres` selection PyMOL uses. Read them off the picture: how many are there, and what
kind are they? Polar, charged, hydrophobic, a recognisable motif?

> **What the pocket looks like:**

**Then type them out**, the way the guide does, as a plain list of residue names. A
binding site is short enough to type, and typing it is what makes you look at each residue
instead of at a number. Keep the list: ex03 and ex04 both build on it.

> **My ligand's contact residues:**

**If your structure is NMR**, there may be no heteroatoms at all, and there will be no
B-factors or occupancies — several rows of the table above will not apply. Say so
explicitly and explain what you can and cannot determine from an NMR ensemble. That is a
real, informative answer, not a failure to complete the task.

**If a species surprises you** — something you cannot classify either way — say so. "I
could not determine whether X is biological, and here is what I checked" is worth more
than a confident guess.

---

## 1. Your protein, and every structure of its family

Start from the chain that binds your ligand (`rcsb_chain` gives its UniProt sequence),
list its family's structures (`search_pdb`, `structure_table`) and draw the guide's two
plots.

**Report:** how many structures, solved how? Where does *yours* sit: method, resolution
against the rest, R-free? You keep your structure: the point is knowing its strengths and
weaknesses.

---

## 2. Visualization

Produce, with a sentence on what each one shows you:

- Your structure coloured by its annotations (from `rcsb_chain`): secondary
  structure, domains, and binding or active sites. Say whose each is (CATH, UniProt, ...);
  one domain is an answer too
- If your structure has a ligand: zoom in on it and its contacts, as the guide's GDP
  view does

You will come back to this section: once you have the alignment from section 4, you add a
**conserved-residue view** to this set.

---

## 3. B-factor analysis: predict first

**Before you run anything:** write your prediction here. Which region of your protein do
you expect to be most flexible, and *why*? Base it on the biology: annotated
sites, domains, termini, loops.

> **My prediction:**
>
> **My reasoning:**

Then plot the B-factors, colour your structure by them, and compare. Were you right? What does the actual pattern
tell you biologically? An honest "I was wrong, and here's what I think I missed" is worth
full marks.

---

## 4. Multiple sequence alignment, and conservation on the structure (required)

Build an alignment, then **put its result back onto your 3D structure**.

**Step 1: build the alignment**, the way the guide does. Take your protein's UniProt
sequence from section 1, find its relatives in Swiss-Prot (`hmmer_search`), align the full sequences (`align`), and
keep one representative per 95% group (`drop_redundant`). Report how many relatives came
back and how many are left. Read their names: one family, or a superfamily, the way FtsZ
reaches tubulin? Keep distant relatives, and say what they are.

**Step 2: predict first, then compute.** Where do you *expect* conservation to cluster: an
active site, a ligand pocket, an interface? Write it down before computing anything.

> **My prediction:**

Then compute conservation (`conservation`) and move it onto your structure's own residue
numbers with the mapping from section 1. Call the most conserved 10% of your
modelled residues conserved, and report the cutoff value that gives.

**Step 3: put it on the structure.** Required: render the conserved residues with your
ligand in the same view, as the guide does for 1FSZ and its GDP, and add that view to your
section 2 figures.

**Step 4: interpret, against a baseline.** Report three numbers: how many of your
ligand's contact residues (from the Character Sheet) are conserved, how many chance would
give (10% of them), and the ratio. On 1FSZ: 8 of 17 against about 2. Matching chance is a
real finding too. Look at the alignment itself (`show_alignment`) as well. For each contact that misses the cutoff, is it a *gap* or a *different
amino acid*? Does conservation coincide with the rigid regions from section 3?

**Judge your own alignment.** Few relatives (under about ten after grouping) make almost
every column look conserved. If that is your case, say so, and weigh your numbers
accordingly.

**Timing:** seconds for the search on EBI's servers, a few more to download and align.

---

## Before you submit

- [ ] Character Sheet filled in (PDB ID, name, organism, why, open question)
- [ ] Residue numbers checked with `renumber_to_uniprot`, and its file used throughout
      if it changed anything
- [ ] Both **predictions written before** their analyses (the B-factor prediction in
      section 3, the conservation prediction in section 4)
- [ ] **Conserved residues rendered on your structure**, added to your section 2 figures
- [ ] The **three conservation numbers**: observed, expected by chance, ratio
- [ ] Every section has a **written interpretation**, not just output
- [ ] Domain boundaries and any external claims are **cited**
- [ ] Anything that didn't work is **described honestly** rather than deleted — a
      documented dead end earns marks, a silently missing section does not